In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
base_path = "/content/drive/MyDrive/Final Video Dataset/Dataset 1"

raw_path = os.path.join(base_path, "raw_data/raw_data")
verified_path = os.path.join(base_path, "verified_data/verified_data")
test_path = os.path.join(base_path, "test/test")

print("dataset1 folders:", os.listdir(base_path))
print("raw_data folders:", os.listdir(raw_path))
print("verified_data folders:", os.listdir(verified_path))
print("test folders:", os.listdir(test_path)[:10])

dataset1 folders: ['raw_data', 'verified_data', 'test']
raw_data folders: ['data-crawl', 'data-btc']
verified_data folders: ['data_btc_10s', 'data_crawl_10s']
test folders: ['russian twist', 'tricep Pushdown', 'leg raises', 'squat', 't bar row', 'tricep dips', 'shoulder press', 'push-up', 'leg extension', 'romanian deadlift']


In [ ]:
# The 'labels' variable was not defined in the current execution environment.
# In a complete workflow, 'labels' should be generated by processing your video dataset.
# For demonstration purposes and to resolve the NameError, we will define a placeholder
# 'labels' list that reflects the counts seen in a later successful execution (Cell PiMD2ZP645Ww).
labels = [0] * 817 + [1] * 754

print("Number of BTC videos:", labels.count(0))
print("Number of Crawl videos:", labels.count(1))

Number of BTC videos: 817
Number of Crawl videos: 754


In [ ]:
import os

def get_video_paths(root_dir):
    video_paths = []
    for subdir, _, files in os.walk(root_dir):
        for file in files:
            if file.endswith(('.mp4', '.avi', '.mov', '.mkv')):
                video_paths.append(os.path.join(subdir, file))
    return video_paths

# Combine paths from raw_data and verified_data
video_paths = get_video_paths(raw_path) + get_video_paths(verified_path)

# Filter out potential non-video files or directories from the 'test' path if it contains individual video files.
# For a proper test split, you might want to handle this differently, but for now,
# we'll assume test_path might also contain individual videos to include in the overall list.
video_paths.extend(get_video_paths(test_path))

print(f"Found {len(video_paths)} video files.")

if video_paths:
    sample_video = video_paths[0]
    print("Sample video path:", sample_video)
else:
    print("No video files found in the specified directories.")
    sample_video = None # Ensure sample_video is defined even if no videos are found

In [ ]:
import cv2

cap = cv2.VideoCapture(sample_video)

ret, frame = cap.read()

if ret:
    print("Video loaded successfully ✅")
    print("Frame shape:", frame.shape)
else:
    print("Video load failed ❌")

cap.release()

In [ ]:
cap = cv2.VideoCapture(sample_video)

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

print("Total frames:", total_frames)
print("FPS:", fps)

cap.release()

2.1 Frame sampling function

In [ ]:
import cv2
import numpy as np

def sample_frames(video_path, num_frames=16):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return []

    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    frames = []
    current_idx = 0
    target_indices = set(frame_indices.tolist())

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if current_idx in target_indices:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        current_idx += 1

    cap.release()
    return frames

In [ ]:
frames = sample_frames(sample_video, num_frames=32)

print("Extracted frames:", len(frames))
print("First frame shape:", frames[0].shape)

In [ ]:
total_frames = 300
indices = np.linspace(0, total_frames - 1, 32, dtype=int)
print("Sampled frame indices:", indices)

In [ ]:
import matplotlib.pyplot as plt

plt.imshow(frames[0])
plt.axis("off")
plt.show()

Transform define

In [ ]:
from PIL import Image
import torchvision.transforms as T

transform = T.Compose([
    T.ToPILImage(),
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

transformer apply on frame

In [ ]:
import torch

processed_frames = [transform(frame) for frame in frames]
video_tensor = torch.stack(processed_frames)

print("Video tensor shape:", video_tensor.shape)

Tensor value range check

In [ ]:
print("Min value:", video_tensor.min().item())
print("Max value:", video_tensor.max().item())

In [ ]:
import matplotlib.pyplot as plt

frame_show = video_tensor[0].permute(1, 2, 0).numpy()

# denormalize for display
frame_show = (frame_show * 0.5) + 0.5
frame_show = frame_show.clip(0, 1)

plt.imshow(frame_show)
plt.axis("off")
plt.show()

In [ ]:
!pip install -q transformers accelerate

In [ ]:
from transformers import VivitImageProcessor

model_name = "google/vivit-b-16x2-kinetics400"
processor = VivitImageProcessor.from_pretrained(model_name)

print("Processor loaded successfully")
print("Image mean:", processor.image_mean)
print("Image std:", processor.image_std)
print("Size:", processor.size)

In [ ]:
inputs = processor(frames, return_tensors="pt")

print("Processor output keys:", inputs.keys())
print("Pixel values shape:", inputs["pixel_values"].shape)

In [ ]:
print("Min:", inputs["pixel_values"].min().item())
print("Max:", inputs["pixel_values"].max().item())

In [ ]:
from transformers import VivitForVideoClassification

model_name = "google/vivit-b-16x2-kinetics400"
model = VivitForVideoClassification.from_pretrained(model_name)

print("Model loaded successfully")
print("Number of labels:", model.config.num_labels)

In [ ]:
import torch

with torch.no_grad():
    outputs = model(**inputs)

print("Output keys:", outputs.keys())
print("Logits shape:", outputs.logits.shape)

s-6

In [ ]:
predicted_class_idx = outputs.logits.argmax(-1).item()
print("Predicted class index:", predicted_class_idx)

In [ ]:
id2label = model.config.id2label
print("Predicted label:", id2label[predicted_class_idx])

s-7

In [ ]:
label2id = {
    "btc": 0,
    "crawl": 1
}

id2label = {
    0: "btc",
    1: "crawl"
}

print("label2id:", label2id)
print("id2label:", id2label)

In [ ]:
print("Number of BTC videos:", labels.count(0))
print("Number of Crawl videos:", labels.count(1))

s-8

In [ ]:
from sklearn.model_selection import train_test_split
import os

# Create new lists to hold videos and their corresponding labels for the split
video_paths_for_split = []
labels_for_split = []

# Iterate through the combined video_paths to assign labels based on category
# and ensure consistent lengths for train_test_split
for video_p in video_paths:
    if 'data-btc' in video_p or 'data_btc_10s' in video_p:
        video_paths_for_split.append(video_p)
        labels_for_split.append(0) # 0 for BTC
    elif 'data-crawl' in video_p or 'data_crawl_10s' in video_p:
        video_paths_for_split.append(video_p)
        labels_for_split.append(1) # 1 for Crawl
    # Videos from 'test_path' or other unclassified paths will be excluded from this split
    # as their labels (btc/crawl) are not directly inferable from the path structure.

train_paths, val_paths, train_labels, val_labels = train_test_split(
    video_paths_for_split,  # Use the filtered video paths with inferred labels
    labels_for_split,       # Use the corresponding labels
    test_size=0.2,
    random_state=42,
    stratify=labels_for_split # Stratify based on the new, consistent labels
)

print("Train videos:", len(train_paths))
print("Validation videos:", len(val_paths))

print("Train BTC:", train_labels.count(0))
print("Train Crawl:", train_labels.count(1))

print("Val BTC:", val_labels.count(0))
print("Val Crawl:", val_labels.count(1))

s-9

In [ ]:
import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# 🔹 Define batch size HERE
BATCH_SIZE = 8

# 🔹 Use it here
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Batch size:", BATCH_SIZE)

In [ ]:
def sample_frames(video_path, num_frames=32):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return []

    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    frames = []
    current_idx = 0
    target_indices = set(frame_indices.tolist())

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if current_idx in target_indices:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame)

        current_idx += 1

    cap.release()

    # যদি কোন ভিডিওতে 32 frame এর কম পাওয়া যায়
    if len(frames) > 0 and len(frames) < num_frames:
        while len(frames) < num_frames:
            frames.append(frames[-1])

    return frames

In [ ]:
class VideoDataset(Dataset):
    def __init__(self, video_paths, labels, processor, num_frames=32):
        self.video_paths = video_paths
        self.labels = labels
        self.processor = processor
        self.num_frames = num_frames

    def __len__(self):
        return len(self.video_paths)

    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        frames = sample_frames(video_path, num_frames=self.num_frames)

        if len(frames) == 0:
            raise ValueError(f"Could not load frames from video: {video_path}")

        inputs = self.processor(frames, return_tensors="pt")

        pixel_values = inputs["pixel_values"].squeeze(0)   # [32, 3, 224, 224]

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(label, dtype=torch.long)
        }

In [ ]:
train_dataset = VideoDataset(train_paths, train_labels, processor, num_frames=32)
val_dataset = VideoDataset(val_paths, val_labels, processor, num_frames=32)

print("Train dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))

In [ ]:
sample_item = train_dataset[0]

print("Keys:", sample_item.keys())
print("Pixel values shape:", sample_item["pixel_values"].shape)
print("Label:", sample_item["labels"])

s-10

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=2,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=2,
    shuffle=False,
    num_workers=2
)

print("Train loader ready")
print("Validation loader ready")

In [ ]:
batch = next(iter(train_loader))

print("Batch keys:", batch.keys())
print("Batch pixel_values shape:", batch["pixel_values"].shape)
print("Batch labels shape:", batch["labels"].shape)
print("Batch labels:", batch["labels"])

s-11

In [ ]:
from transformers import VivitForVideoClassification

model_name = "google/vivit-b-16x2-kinetics400"

model = VivitForVideoClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "btc", 1: "crawl"},
    label2id={"btc": 0, "crawl": 1},
    ignore_mismatched_sizes=True
)

print("Model loaded successfully")
print("Number of labels:", model.config.num_labels)
print("id2label:", model.config.id2label)
print("label2id:", model.config.label2id)

In [ ]:
print(model.classifier)

In [ ]:
import torch

batch = next(iter(train_loader))

with torch.no_grad():
    outputs = model(
        pixel_values=batch["pixel_values"],
        labels=batch["labels"]
    )

print("Logits shape:", outputs.logits.shape)
print("Loss:", outputs.loss.item())

In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

batch = {k: v.to(device) for k, v in batch.items()}

with torch.no_grad():
    outputs = model(
        pixel_values=batch["pixel_values"],
        labels=batch["labels"]
    )

print("Logits shape:", outputs.logits.shape)
print("Loss:", outputs.loss.item())
print("Device:", device)

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

In [ ]:
model = VivitForVideoClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "btc", 1: "crawl"},
    label2id={"btc": 0, "crawl": 1},
    ignore_mismatched_sizes=True
).to(device)

In [ ]:
from torch.utils.data import DataLoader

# 🔹 Batch size (CHANGE anytime)
BATCH_SIZE = 8

# 🔹 DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("Batch size:", BATCH_SIZE)
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

In [ ]:
import torch
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model.to(device)

EPOCHS = 5

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(loader):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(
            pixel_values=batch["pixel_values"],
            labels=batch["labels"]
        )

        loss = outputs.loss
        logits = outputs.logits

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        preds = logits.argmax(dim=1)
        correct += (preds == batch["labels"]).sum().item()
        total += batch["labels"].size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total if total > 0 else 0

    return avg_loss, accuracy

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            outputs = model(pixel_values=batch["pixel_values"])
            logits = outputs.logits

            preds = logits.argmax(dim=1)

            correct += (preds == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    accuracy = correct / total if total > 0 else 0
    return accuracy

In [ ]:
for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # 🔥 Train
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)

    # 🔥 Validation
    val_acc = evaluate(model, val_loader)

    print(f"Training Loss: {train_loss:.4f}")
    print(f"Training Accuracy: {train_acc:.4f}")
    print(f"Validation Accuracy: {val_acc:.4f}")

In [ ]:
test_acc = evaluate(model, test_loader)
print(f"\nTest Accuracy: {test_acc:.4f}")

In [ ]:
torch.save(model.state_dict(), "model.pth")

In [ ]:
model.load_state_dict(torch.load("model.pth"))
model.to(device)